# Project Setup

**Run this notebook once before running any experiments.**

It does four things:

1. Verifies the Python environment has all required packages at compatible versions.
2. Confirms the raw data files are present in `data/raw/`.
3. Regenerates the train/test splits from raw data using `src/data/prepare.py`. This ensures the splits are always derived from the raw source rather than relying on committed CSV files that could drift out of sync.
4. Validates the generated splits and lists available experiments, confirming the project is ready to use.

> **If any cell fails**, read the error message — it will tell you exactly what is missing. Common issues are a missing package (fix: `conda install <package>`) or a missing raw data file (fix: check `data/raw/` contains `batch1_raw.csv` and `batch2_raw.csv`).

## Step 1 — Environment check

Checks that all required packages are importable and prints their versions for the record.
Version information here should be included in the dissertation's reproducibility appendix.

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib

REQUIRED = [
    ("numpy",      "numpy"),
    ("pandas",     "pandas"),
    ("scipy",      "scipy"),
    ("sklearn",    "scikit-learn"),
    ("xgboost",    "xgboost"),
    ("optuna",     "optuna"),
    ("matplotlib", "matplotlib"),
    ("seaborn",    "seaborn"),
    ("yaml",       "PyYAML"),
]

all_ok = True
print(f"Python {sys.version}\n")
print(f"{'Package':<14} {'Version':<12} Status")
print("-" * 40)
for import_name, display_name in REQUIRED:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"{display_name:<14} {version:<12} OK")
    except ImportError:
        print(f"{display_name:<14} {'MISSING':<12} FAIL")
        all_ok = False

print()
if all_ok:
    print("All packages present.")
else:
    raise EnvironmentError(
        "One or more required packages are missing. "
        "Install them with: conda install <package>  or  pip install <package>"
    )

## Step 2 — Validate raw data

Checks that the raw batch CSVs exist in `data/raw/`. These files are the source of truth for all splits and must be present before the prepare step can run.

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"

REQUIRED_RAW = [
    "batch1_raw.csv",
    "batch2_raw.csv",
]

all_ok = True
print(f"Raw data directory: {RAW_DIR}\n")
for fname in REQUIRED_RAW:
    path = RAW_DIR / fname
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  {fname:<30} {size_kb:>7.1f} KB   OK")
    else:
        print(f"  {fname:<30} {'MISSING':>10}   FAIL")
        all_ok = False

# List any other files present for reference (Excel source, etc.)
extras = [f for f in RAW_DIR.iterdir() if f.name not in REQUIRED_RAW and not f.name.startswith(".")]
if extras:
    print()
    print("Other files in data/raw/:")
    for f in sorted(extras):
        print(f"  {f.name}")

print()
if not all_ok:
    raise FileNotFoundError(
        f"Required raw data files are missing from {RAW_DIR}. "
        "Place batch1_raw.csv and batch2_raw.csv there before continuing."
    )
print("Raw data OK.")

## Step 3 — Generate train/test splits

Runs the full data preparation pipeline:

1. Reads both raw batch CSVs and combines them.
2. Expands the categorical `Formulation` column (F1/F2/F3) into numeric component percentages.
3. Constructs a `Composite_Mix_ID` group label so that replicate measurements of the same formulation are never split across train and test.
4. Performs an 80/20 group-level train/test split and writes the results to `data/processed/splits/`.

Re-running this cell with the same seed produces identical splits. Change `SEED` only if you intentionally want a different split, and document the change.

In [ ]:
from src.data.prepare import build_combined, expand_formulation, group_split

SEED = 42        # must match cfg.seed in base.yaml
TEST_SIZE = 0.2  # must match the value used when the project was originally set up

OUT_DIR = PROJECT_ROOT / "data" / "processed"

print("Combining raw batches...")
combined = build_combined(RAW_DIR / "batch1_raw.csv", RAW_DIR / "batch2_raw.csv")
print(f"  Combined: {len(combined)} rows, {combined['Composite_Mix_ID'].nunique()} unique mixes")

print("Expanding formulation labels...")
expanded = expand_formulation(combined)
expanded.to_csv(OUT_DIR / "combined_slurry_data_expanded.csv", index=False)
print(f"  Saved combined_slurry_data_expanded.csv ({len(expanded)} rows)")

print("Splitting...")
meta = group_split(expanded, OUT_DIR / "splits", test_size=TEST_SIZE, seed=SEED)
print(f"  Train: {meta['train_rows']} rows across {meta['train_groups']} groups")
print(f"  Test:  {meta['test_rows']} rows across {meta['test_groups']} groups")
print(f"  Seed:  {meta['seed']}")
print()
print("Splits written to data/processed/splits/")

## Step 4 — Validate splits and confirm project is ready

Loads the generated splits through the same code path used by every experiment, verifies the no-leakage invariant, and lists all available experiment configs.

In [ ]:
from src.config import load_config, list_experiments
from src.data.splits import load_splits

# Load via a real config so the same code path as experiments is exercised.
cfg = load_config("base")
train_df, test_df = load_splits(cfg)

print("Split validation")
print("-" * 40)
print(f"  Train rows  : {len(train_df)}")
print(f"  Test rows   : {len(test_df)}")
print(f"  Train groups: {train_df['Composite_Mix_ID'].nunique()}")
print(f"  Test groups : {test_df['Composite_Mix_ID'].nunique()}")
overlap = set(train_df['Composite_Mix_ID'].unique()) & set(test_df['Composite_Mix_ID'].unique())
print(f"  Group overlap: {len(overlap)} (should be 0)")
print(f"  Columns     : {list(train_df.columns)}")
print()

experiments = list_experiments()
print(f"Available experiments ({len(experiments)}):")
for name in experiments:
    exp_cfg = load_config(name)
    print(f"  {name:<35} sections={exp_cfg.run.sections}")

print()
print("Project is ready. Open 01_run_experiment.ipynb to run an experiment.")